# Navarasa Image Classification -- EfficientNet-B2 Transfer Learning
Production-ready training pipeline for 9 Indian aesthetic emotions (Rasas).

In [ ]:
# Zaroori packages install karo
!pip install -q torch torchvision matplotlib seaborn scikit-learn tqdm

## Configuration & Environment Setup

In [ ]:
import os, random, copy, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_B2_Weights
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.notebook import tqdm
from PIL import Image

import time

def resilient_pil_loader(path, retries=3, delay=2):
    """PIL loader with retry logic for Google Drive disconnects."""
    for attempt in range(retries):
        try:
            with open(path, 'rb') as f:
                img = Image.open(f)
                return img.convert('RGB')
        except OSError as e:
            if attempt < retries - 1:
                print(f"  [WARN] Read failed for {os.path.basename(path)}, retrying in {delay}s... ({e})")
                time.sleep(delay)
                # Try remounting drive
                try:
                    from google.colab import drive
                    drive.mount('/content/drive', force_remount=True)
                except Exception:
                    pass
            else:
                raise

warnings.filterwarnings('ignore')

# Saare hyperparameters ek jagah define karo
CONFIG = {
    "batch_size": 32,
    "num_workers": 0,
    "num_classes": 9,
    "seed": 42,
    "phase1_epochs": 8,
    "phase2_epochs": 32,
    "head_lr": 1e-3,
    "backbone_lr": 1e-5,
    "classifier_lr": 1e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.1,
    "early_stop_patience": 7,
    "overfit_gap_threshold": 15.0,
    "target_val_acc": 85.0,
    "train_dir": "/content/drive/MyDrive/train_new",
    "valid_dir": "/content/drive/MyDrive/valid_new",
    "test_dir": "/content/drive/MyDrive/test_new",
    "save_dir": "/content/drive/MyDrive/navarasa_results_new"
}

# Reproducibility ke liye seed fix karo
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])

training_complete = False

# GPU milega toh GPU varna CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## Google Drive Mount

In [ ]:
# Colab me Drive mount karo
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Results save karne ka folder banao
os.makedirs(CONFIG["save_dir"], exist_ok=True)
print(f"Save directory ready: {CONFIG['save_dir']}")

Mounted at /content/drive
Save directory ready: /content/drive/MyDrive/navarasa_results_new


## Data Loading & Preprocessing

In [ ]:
# Training ke liye heavy augmentation lagao
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Test ke liye sirf resize aur crop
val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ImageFolder se load karo -- folder name = class label
train_dataset = datasets.ImageFolder(CONFIG["train_dir"], transform=train_transform, loader=resilient_pil_loader)
val_dataset = datasets.ImageFolder(CONFIG["valid_dir"], transform=val_test_transform, loader=resilient_pil_loader)
test_dataset = datasets.ImageFolder(CONFIG["test_dir"], transform=val_test_transform, loader=resilient_pil_loader)

class_names = train_dataset.classes

rasa_to_bhava = {
    'adhbut': 'Adhbut',
    'bhayank': 'Bhayanak',
    'bhayanak': 'Bhayanak',
    'bhibhatsa': 'Bhibhatsa',
    'hasya': 'Hasya',
    'karuna': 'Karuna',
    'raudra': 'Raudra',
    'shanta': 'Shanta',
    'shringara': 'Shringara',
    'veer': 'Veer'
}
class_names = [rasa_to_bhava.get(name.lower(), name) for name in class_names]
print(f"Classes: {class_names}")
print(f"Class to index: {train_dataset.class_to_idx}")
print(f"Dataset sizes -- Train: {len(train_dataset)} | Valid: {len(val_dataset)} | Test: {len(test_dataset)}")

# DataLoader banao
pin = device.type == "cuda"
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                          shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=pin)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"],
                        shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"],
                         shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=pin)

Classes: ['adhbuta', 'Bhayanak', 'Bhibhatsa', 'Hasya', 'Karuna', 'Raudra', 'Shanta', 'Shringara', 'Veer']
Class to index: {'adhbuta': 0, 'bhayanak': 1, 'bhibhatsa': 2, 'hasya': 3, 'karuna': 4, 'raudra': 5, 'shanta': 6, 'shringara': 7, 'veer': 8}
Dataset sizes -- Train: 14805 | Valid: 3635 | Test: 3312


## Sample Training Images

In [ ]:
# Denormalize karke asli image dikhao
def denormalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    tensor = tensor.clone()
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return torch.clamp(tensor, 0, 1)

# 16 sample images grid me dikhao
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.subplots_adjust(wspace=0.4, hspace=0.4) # Add space around blocks visually
fig.suptitle("Sample Training Images (Every 10th Image)", fontsize=16, fontweight='bold')
for i, ax in enumerate(axes.flat):
    if i < 16:
        img_tensor, label = train_dataset[i * 10] # Skip 10 frames to avoid continuous same pic
        img = denormalize(img_tensor).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(class_names[label], fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["save_dir"], "sample_images_new.png"), dpi=150, bbox_inches='tight')
plt.show()

## Model Architecture -- EfficientNet-B2

In [ ]:
# Pretrained EfficientNet-B2 load karo
model = models.efficientnet_b2(weights=EfficientNet_B2_Weights.IMAGENET1K_V1)

# Pehle saari layers freeze karo
for param in model.parameters():
    param.requires_grad = False

# Custom classifier head lagao
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features=1408, out_features=512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, CONFIG["num_classes"])
)

model = model.to(device)

# Parameters gino
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")
# Resume logic
import os
checkpoint_path = os.path.join(CONFIG["save_dir"], "best_navarasa_model_new.pth")
start_phase = 1
start_epoch_p1 = 1
start_epoch_p2 = 1
loaded_best_val_acc = 0.0

if os.path.exists(checkpoint_path):
    print(f"\n--- FOUND EXISTING CHECKPOINT: {checkpoint_path} ---")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    saved_epoch = checkpoint['epoch']
    loaded_best_val_acc = checkpoint.get('val_acc', 0.0)

    if saved_epoch <= CONFIG["phase1_epochs"]:
        start_phase = 1
        start_epoch_p1 = saved_epoch + 1
        print(f"Resuming from Phase 1, Epoch {start_epoch_p1} (Best Val Acc: {loaded_best_val_acc:.2f}%)")
    else:
        start_phase = 2
        start_epoch_p2 = saved_epoch - CONFIG["phase1_epochs"] + 1
        print(f"Resuming from Phase 2, Local Epoch {start_epoch_p2} (Global Epoch {saved_epoch + 1})")
else:
    print("\nNo checkpoint found. Starting from scratch.")

    # If we already achieved target accuracy, skip retraining entirely
    if loaded_best_val_acc >= CONFIG["target_val_acc"]:
        training_complete = True
        print(f"  -> Training already completed! Best val_acc ({loaded_best_val_acc:.2f}%) >= target ({CONFIG['target_val_acc']}%). Skipping to evaluation.")
    else:
        training_complete = False


Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 121MB/s] 


Total parameters: 8,427,019
Trainable parameters: 726,025
Frozen parameters: 7,700,994

--- FOUND EXISTING CHECKPOINT: /content/drive/MyDrive/navarasa_results_new/best_navarasa_model_new.pth ---
Resuming from Phase 2, Local Epoch 22 (Global Epoch 30)


## Training & Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    # Ek epoch train karo, loss aur accuracy return karo
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc="Training", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100.0*correct/total:.1f}%")

    return running_loss / total, 100.0 * correct / total


def evaluate(model, loader, criterion, device):

    # Validation/Test set pe evaluate karo
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        pbar = tqdm(loader, desc="Evaluating", leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, 100.0 * correct / total

## Phase 1 -- Head Warmup (Backbone Frozen)

In [ ]:
# Phase 1 -- Sirf classifier head train karo
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
optimizer_p1 = optim.Adam(model.classifier.parameters(), lr=CONFIG["head_lr"],
                          weight_decay=CONFIG["weight_decay"])
scheduler_p1 = optim.lr_scheduler.CosineAnnealingLR(optimizer_p1, T_max=CONFIG["phase1_epochs"])

# History lists
train_losses, val_losses = [], []
train_accs, val_accs = [], []
lr_history = []
best_val_acc = loaded_best_val_acc
best_model_state = None
overfit_counter = 0

print("=" * 80)
if not training_complete:
    print("PHASE 1 -- HEAD WARMUP")
print("=" * 80)

if start_phase == 1:
    for epoch in range(start_epoch_p1, CONFIG["phase1_epochs"] + 1):
        current_lr = optimizer_p1.param_groups[0]['lr']
        lr_history.append(current_lr)

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer_p1, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        scheduler_p1.step()

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        gap = train_acc - val_acc

        # Overfitting check
        if gap > CONFIG["overfit_gap_threshold"]:
            overfit_counter += 1
        else:
            overfit_counter = 0
        if overfit_counter >= 3:
            print(f"WARNING: Overfitting detected -- train/val gap > {CONFIG['overfit_gap_threshold']}% for 3 consecutive epochs")

        # Best model save karo
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer_p1.state_dict(),
                'val_acc': best_val_acc,
                'class_names': class_names
            }, os.path.join(CONFIG["save_dir"], "best_navarasa_model_new.pth"))

        print(f"Epoch [{epoch}/{CONFIG['phase1_epochs']}] | Phase 1 | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
              f"Gap: {gap:.2f}% | LR: {current_lr:.6f}")

else:
    print("\nSkipping Phase 1 as checkpoint is already in Phase 2.")
print(f"\nPhase 1 complete. Best val_acc: {best_val_acc:.2f}%. Proceeding to Phase 2.")

PHASE 1 -- HEAD WARMUP

Skipping Phase 1 as checkpoint is already in Phase 2.

Phase 1 complete. Best val_acc: 87.79%. Proceeding to Phase 2.


## Phase 2 -- Full Fine-tuning (All Layers Unfrozen)

In [ ]:
# Phase 2 -- Saari layers unfreeze karo
for param in model.parameters():
    param.requires_grad = True

# Differential LR -- backbone slow, classifier fast
backbone_params, classifier_params = [], []
for name, param in model.named_parameters():
    if "classifier" in name:
        classifier_params.append(param)
    else:
        backbone_params.append(param)

optimizer_p2 = optim.AdamW([
    {"params": backbone_params, "lr": CONFIG["backbone_lr"]},
    {"params": classifier_params, "lr": CONFIG["classifier_lr"]}
], weight_decay=CONFIG["weight_decay"])

scheduler_p2 = optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=24, eta_min=1e-7)

patience_counter = 0
val_loss_increase_counter = 0
prev_val_loss = float('inf')
overfit_counter = 0

trainable_p2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 2 trainable parameters: {trainable_p2:,}")
print("=" * 80)
print("PHASE 2 -- FULL FINE-TUNING (24 epochs to reach epoch 32 total)")
print("=" * 80)

phase2_max_epochs = 24

for epoch in range(1, phase2_max_epochs + 1):
    global_epoch = CONFIG["phase1_epochs"] + epoch
    current_lr = optimizer_p2.param_groups[0]['lr']
    lr_history.append(current_lr)

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer_p2, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    scheduler_p2.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    gap = train_acc - val_acc

    # Overfitting check
    if gap > CONFIG["overfit_gap_threshold"]:
        overfit_counter += 1
    else:
        overfit_counter = 0
    if overfit_counter >= 3:
        print(f"WARNING: Overfitting detected -- train/val gap > {CONFIG['overfit_gap_threshold']}% for 3 consecutive epochs")

    # Val loss badhta ja raha hai toh stop karo
    if val_loss >= prev_val_loss:
        val_loss_increase_counter += 1
    else:
        val_loss_increase_counter = 0
    prev_val_loss = val_loss

    if val_loss_increase_counter >= 5:
        print("Early stopping triggered -- val_loss increased for 5 consecutive epochs")
        break

    # Best model update karo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save({
            'epoch': global_epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer_p2.state_dict(),
            'val_acc': best_val_acc,
            'class_names': class_names
        }, os.path.join(CONFIG["save_dir"], "best_navarasa_model_new.pth"))
    else:
        patience_counter += 1

    if patience_counter >= CONFIG["early_stop_patience"]:
        print(f"Early stopping -- val_acc improved nahi {CONFIG['early_stop_patience']} epochs se")
        break

    print(f"Epoch [{global_epoch}/32] | Phase 2 | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
          f"Gap: {gap:.2f}% | LR: {current_lr:.2e}")

# Final model bhi save karo
torch.save({
    'epoch': global_epoch,
    'model_state_dict': model.state_dict(),
    'val_acc': val_acc,
    'class_names': class_names
}, os.path.join(CONFIG["save_dir"], "final_navarasa_model.pth"))

# Save training history for resume in case
import json as _json
history = {
    'train_losses': train_losses,
    'val_losses': val_losses,
    'train_accs': train_accs,
    'val_accs': val_accs,
    'lr_history': lr_history
}
with open(os.path.join(CONFIG["save_dir"], "training_history.json"), 'w') as _hf:
    _json.dump(history, _hf)

# Accuracy target check karo
if best_val_acc < 80.0:
    print("Accuracy below target. Consider: (1) more epochs, (2) additional augmentation, (3) larger backbone (EfficientNet-B4).")
elif best_val_acc < CONFIG["target_val_acc"]:
    print(f"Val accuracy {best_val_acc:.2f}% -- close to target but not there yet.")
else:
    print(f"Training converged successfully. Convergence Target achieved! Best val_acc: {best_val_acc:.2f}%")


Phase 2 trainable parameters: 8,427,019
PHASE 2 -- FULL FINE-TUNING (24 epochs to reach epoch 32 total)


Training:   0%|          | 0/463 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Training History Plots

In [ ]:
# Load training history if skipped training (resume mode)
import json as _json
import matplotlib.pyplot as plt

history_path = os.path.join(CONFIG["save_dir"], "training_history.json")
if len(train_losses) == 0 and os.path.exists(history_path):
    with open(history_path, 'r') as _hf:
        history = _json.load(_hf)
    train_losses = history['train_losses']
    val_losses = history['val_losses']
    train_accs = history['train_accs']
    val_accs = history['val_accs']
    lr_history = history['lr_history']
    print(f"Loaded training history: {len(train_losses)} epochs")
elif len(train_losses) == 0:
    print("WARNING: No training history found. Plots will be empty.")

# Print data availability
print(f"\nData points available:")
print(f"  Train losses: {len(train_losses)}")
print(f"  Val losses: {len(val_losses)}")
print(f"  Train accs: {len(train_accs)}")
print(f"  Val accs: {len(val_accs)}")
print(f"  LR history: {len(lr_history)}")

# Check there's data available for plotting
if len(train_losses) > 0 and len(val_losses) > 0:
    # Loss curve plot
    fig, ax = plt.subplots(figsize=(10, 6))
    epochs = list(range(1, len(train_losses)+1))
    ax.plot(epochs, train_losses, label='Train Loss', linewidth=2, marker='o', markersize=3)
    ax.plot(epochs, val_losses, label='Val Loss', linewidth=2, marker='s', markersize=3)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["save_dir"], "loss_curve_new.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print("Loss curve saved")
else:
    print("Skipping loss curve - insufficient data")

if len(train_accs) > 0 and len(val_accs) > 0:
    # Accuracy curve plot
    fig, ax = plt.subplots(figsize=(10, 6))
    epochs = list(range(1, len(train_accs)+1))
    ax.plot(epochs, train_accs, label='Train Acc', linewidth=2, marker='o', markersize=3)
    ax.plot(epochs, val_accs, label='Val Acc', linewidth=2, marker='s', markersize=3)
    ax.axhline(y=CONFIG["target_val_acc"], color='r', linestyle='--', linewidth=2,
               label=f"Target ({CONFIG['target_val_acc']}%)")
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Accuracy (%)', fontsize=12)
    ax.set_title('Training vs Validation Accuracy', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["save_dir"], "accuracy_curve_new.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print("Accuracy curve saved")
else:
    print("Skipping accuracy curve - insufficient data")

if len(lr_history) > 0:
    # LR schedule plot
    fig, ax = plt.subplots(figsize=(10, 6))
    epochs = list(range(1, len(lr_history)+1))
    ax.plot(epochs, lr_history, linewidth=2, color='green', marker='o', markersize=3)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Learning Rate', fontsize=12)
    ax.set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["save_dir"], "lr_schedule_new.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print("LR schedule saved")
else:
    print("Skipping LR schedule - insufficient data")

if len(train_accs) > 0 and len(val_accs) > 0 and len(train_accs) == len(val_accs):
    # Overfitting gap plot
    gaps = [t - v for t, v in zip(train_accs, val_accs)]
    fig, ax = plt.subplots(figsize=(10, 6))
    epochs = list(range(1, len(gaps)+1))
    ax.plot(epochs, gaps, linewidth=2, color='orange', marker='o', markersize=3, label='Overfitting Gap')
    ax.axhline(y=CONFIG["overfit_gap_threshold"], color='r', linestyle='--', linewidth=2,
               label=f"Overfit threshold ({CONFIG['overfit_gap_threshold']}%)")
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Train Acc - Val Acc (%)', fontsize=12)
    ax.set_title('Overfitting Gap per Epoch', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    if len(gaps) > 0:
        gap_min = min(gaps)
        gap_max = max(gaps)
        threshold = CONFIG["overfit_gap_threshold"]
        y_min = min(gap_min, threshold) - 2
        y_max = max(gap_max, threshold) + 2
        ax.set_ylim(y_min, y_max)

    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["save_dir"], "overfitting_gap_new.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print("Overfitting gap plot saved")
    if len(gaps) > 0:
        print(f"  Gap range: {min(gaps):.2f}% to {max(gaps):.2f}%")
else:
    print("Skipping overfitting gap plot - insufficient data")

print("\nAll available plots generated successfully")


Loaded training history: 24 epochs

Data points available:
  Train losses: 24
  Val losses: 24
  Train accs: 24
  Val accs: 24
  LR history: 25
Loss curve saved
Accuracy curve saved
LR schedule saved
Overfitting gap plot saved
  Gap range: -1.34% to 0.96%

All available plots generated successfully


## Test Set Evaluation

In [ ]:
# Best model load karo evaluation ke liye
checkpoint = torch.load(os.path.join(CONFIG["save_dir"], "best_navarasa_model.pth"), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Best model loaded -- Epoch: {checkpoint['epoch']}, Val Acc: {checkpoint['val_acc']:.2f}%")

# Test set pe evaluate karo
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"\nOverall Test Accuracy: {test_acc:.2f}%")

# Per class accuracy nikalo
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Test inference", leave=False):
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print("\nPer-class accuracy:")
for i, name in enumerate(class_names):
    mask = all_labels == i
    acc = 100.0 * (all_preds[mask] == all_labels[mask]).sum() / mask.sum()
    print(f"  {name}: {acc:.2f}%")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

Best model loaded -- Epoch: 28, Val Acc: 93.43%


Evaluating:   0%|          | 0/104 [00:00<?, ?it/s]


Overall Test Accuracy: 87.32%


Test inference:   0%|          | 0/104 [00:00<?, ?it/s]


Per-class accuracy:
  adhbuta: 79.54%
  Bhayanak: 80.10%
  Bhibhatsa: 83.76%
  Hasya: 88.46%
  Karuna: 83.43%
  Raudra: 92.02%
  Shanta: 94.13%
  Shringara: 90.42%
  Veer: 93.04%

Classification Report:
              precision    recall  f1-score   support

     adhbuta       0.93      0.80      0.86       303
    Bhayanak       0.82      0.80      0.81       397
   Bhibhatsa       0.86      0.84      0.85       425
       Hasya       0.94      0.88      0.91       364
      Karuna       0.88      0.83      0.85       338
      Raudra       0.84      0.92      0.88       426
      Shanta       0.98      0.94      0.96       409
   Shringara       0.77      0.90      0.83       334
        Veer       0.87      0.93      0.90       316

    accuracy                           0.87      3312
   macro avg       0.88      0.87      0.87      3312
weighted avg       0.88      0.87      0.87      3312



## Confusion Matrix

In [ ]:
# Confusion matrix banao aur heatmap me dikhao
cm = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names,
            yticklabels=class_names, ax=ax, linewidths=0.5)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Normalized Confusion Matrix -- Navarasa Test Set', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["save_dir"], "confusion_matrix_new.png"), dpi=150, bbox_inches='tight')
plt.show()

## Top-5 Misclassified Images

In [ ]:
# Top 5 galat predictions dhundho aur dikhao
misclassified_indices = np.where(all_preds != all_labels)[0]

# Confidence ke basis pe sort karo -- sabse confident galat predictions
model.eval()
misclassified_data = []
with torch.no_grad():
    for idx in misclassified_indices:
        img, true_label = test_dataset[idx]
        img_input = img.unsqueeze(0).to(device)
        output = model(img_input)
        probs = torch.softmax(output, dim=1)
        pred_conf, pred_label = torch.max(probs, 1)
        misclassified_data.append({
            'idx': idx,
            'true': true_label,
            'pred': pred_label.item(),
            'conf': pred_conf.item(),
            'img': img
        })

misclassified_data.sort(key=lambda x: x['conf'], reverse=True)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle("Top-5 Misclassified Images (Highest Confidence Mistakes)", fontsize=14, fontweight='bold')
for i, ax in enumerate(axes):
    if i < len(misclassified_data):
        item = misclassified_data[i]
        img = denormalize(item['img']).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(f"True: {class_names[item['true']]}\nPred: {class_names[item['pred']]} ({item['conf']*100:.1f}%)",
                     fontsize=9, color='red')
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["save_dir"], "misclassified_top5_new.png"), dpi=150, bbox_inches='tight')
plt.show()

## Inference Function -- Single Image Prediction

In [ ]:
def predict_rasa(image_path, model, class_names, device):
    # Ek image load karo, predict karo, bar chart dikhao
    img = Image.open(image_path).convert('RGB')

    # Validation wali transform lagao
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    img_tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        probs = torch.softmax(output, dim=1).cpu().numpy()[0]

    pred_idx = np.argmax(probs)
    pred_class = class_names[pred_idx]
    pred_conf = probs[pred_idx] * 100.0

    print(f"Predicted Rasa: {pred_class}")
    print(f"Confidence: {pred_conf:.2f}%")

    # Bar chart banao -- top prediction alag color me
    colors = ['#3498db'] * len(class_names)
    colors[pred_idx] = '#e74c3c'

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].imshow(img)
    axes[0].set_title(f"Predicted: {pred_class} ({pred_conf:.1f}%)", fontsize=12)
    axes[0].axis('off')

    bars = axes[1].barh(class_names, probs * 100, color=colors, edgecolor='black', linewidth=0.5)
    axes[1].set_xlabel('Probability (%)')
    axes[1].set_title('Rasa Probabilities')
    axes[1].set_xlim(0, 100)

    for bar, prob in zip(bars, probs):
        axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                     f"{prob*100:.1f}%", va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

    return pred_class, pred_conf

sample_path = test_dataset.samples[0][0]
print(f"Testing on: {sample_path}")
pred_class, pred_conf = predict_rasa(sample_path, model, class_names, device)

Testing on: /content/drive/MyDrive/test_new/adhbut/newiwe (1)_frame_0673.jpg
Predicted Rasa: adhbuta
Confidence: 35.27%


## Summary
Training complete. Best model saved as `best_navarasa_model.pth`. All plots saved to Google Drive.

In [ ]:
# Training ke liye heavy augmentation lagao
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Test ke liye sirf resize aur crop
val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ImageFolder se load karo -- folder name = class label
train_dataset = datasets.ImageFolder(CONFIG["train_dir"], transform=train_transform, loader=resilient_pil_loader)
val_dataset = datasets.ImageFolder(CONFIG["valid_dir"], transform=val_test_transform, loader=resilient_pil_loader)
test_dataset = datasets.ImageFolder(CONFIG["test_dir"], transform=val_test_transform, loader=resilient_pil_loader)

class_names = train_dataset.classes

rasa_to_bhava = {
    'adhbut': 'Adhbut',
    'bhayank': 'Bhayanak',
    'bhayanak': 'Bhayanak',
    'bhibhatsa': 'Bhibhatsa',
    'hasya': 'Hasya',
    'karuna': 'Karuna',
    'raudra': 'Raudra',
    'shanta': 'Shanta',
    'shringara': 'Shringara',
    'veer': 'Veer'
}
class_names = [rasa_to_bhava.get(name.lower(), name) for name in class_names]
print(f"Classes: {class_names}")
print(f"Class to index: {train_dataset.class_to_idx}")
print(f"Dataset sizes -- Train: {len(train_dataset)} | Valid: {len(val_dataset)} | Test: {len(test_dataset)}")

# DataLoader banao
pin = device.type == "cuda"
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                          shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=pin)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"],
                        shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"],
                         shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=pin)

Classes: ['adhbuta', 'Bhayanak', 'Bhibhatsa', 'Hasya', 'Karuna', 'Raudra', 'Shanta', 'Shringara', 'Veer']
Class to index: {'adhbuta': 0, 'bhayanak': 1, 'bhibhatsa': 2, 'hasya': 3, 'karuna': 4, 'raudra': 5, 'shanta': 6, 'shringara': 7, 'veer': 8}
Dataset sizes -- Train: 14805 | Valid: 3635 | Test: 3312


In [ ]:
# ADHBUTA ONLY EVALUATION
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Subset, DataLoader

adhbut_class_idx = None
for idx, name in enumerate(test_dataset.classes):
    if 'adhbut' in name.lower() or 'adhbuta' in name.lower() or 'vismaya' in name.lower():
        adhbut_class_idx = idx
        break

if adhbut_class_idx is None:
    print("ERROR: Could not find adhbut class. Available classes:")
    print(test_dataset.classes)
    raise SystemExit

print(f"Found Adhbuta class at index {adhbut_class_idx}: {class_names[adhbut_class_idx]}")

adhbut_indices = [i for i, (_, label) in enumerate(test_dataset.samples) if label == adhbut_class_idx]
print(f"Total Adhbuta test samples: {len(adhbut_indices)}")

adhbut_subset = Subset(test_dataset, adhbut_indices)
adhbut_loader = DataLoader(adhbut_subset, batch_size=CONFIG["batch_size"], shuffle=False)

model.eval()
all_preds, all_confs = [], []
with torch.no_grad():
    for images, labels in adhbut_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        pred_conf, pred_label = torch.max(probs, 1)
        all_preds.extend(pred_label.cpu().numpy())
        all_confs.extend(pred_conf.cpu().numpy())

all_preds = np.array(all_preds)
all_confs = np.array(all_confs)

correct = (all_preds == adhbut_class_idx).sum()
total = len(all_preds)
acc = 100.0 * correct / total
wrong_mask = all_preds != adhbut_class_idx

print(f"\n{'='*60}")
print(f"ADHBUTA TEST ACCURACY: {acc:.2f}% ({correct}/{total})")
print(f"{'='*60}")

print(f"\nMisclassification Breakdown:")
wrong_preds = all_preds[wrong_mask]
if len(wrong_preds) > 0:
    for cls_idx in range(len(class_names)):
        count = (wrong_preds == cls_idx).sum()
        if count > 0:
            print(f"  Confused as {class_names[cls_idx]}: {count} times ({100.0*count/total:.1f}%)")
else:
    print("  No misclassifications!")

correct_confs = all_confs[~wrong_mask]
wrong_confs = all_confs[wrong_mask]
print(f"\nConfidence Statistics:")
if len(correct_confs) > 0:
    print(f"  Correct predictions avg confidence: {correct_confs.mean()*100:.1f}%")
if len(wrong_confs) > 0:
    print(f"  Wrong predictions avg confidence:   {wrong_confs.mean()*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if len(correct_confs) > 0:
    axes[0].hist(correct_confs * 100, bins=20, color='green', alpha=0.7, label='Correct')
if len(wrong_confs) > 0:
    axes[0].hist(wrong_confs * 100, bins=20, color='red', alpha=0.7, label='Wrong')
axes[0].set_xlabel('Confidence (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('Adhbuta Prediction Confidence Distribution')
axes[0].legend()

pred_counts = np.zeros(len(class_names))
for p in all_preds:
    pred_counts[p] += 1
colors = ['#e74c3c' if i != adhbut_class_idx else '#2ecc71' for i in range(len(class_names))]
axes[1].barh(class_names, pred_counts, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Prediction Count')
axes[1].set_title('Where Adhbuta Samples Get Classified')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["save_dir"], "adhbuta_analysis.png"), dpi=150, bbox_inches='tight')
plt.show()

bhayanak_class_idx = None
for idx, name in enumerate(test_dataset.classes):
    if 'bhayanak' in name.lower() or 'bhaya' in name.lower():
        bhayanak_class_idx = idx
        break

if bhayanak_class_idx is not None:
    bhayanak_mistakes = []
    for i, (pred, conf) in enumerate(zip(all_preds, all_confs)):
        if pred == bhayanak_class_idx:
            real_idx = adhbut_indices[i]
            bhayanak_mistakes.append((i, real_idx, conf))

    bhayanak_mistakes.sort(key=lambda x: -x[2])
    show_count = min(5, len(bhayanak_mistakes))

    if show_count > 0:
        print(f"\n{'='*60}")
        print(f"TOP {show_count} ADHBUTA IMAGES MISCLASSIFIED AS BHAYANAK")
        print(f"{'='*60}")

        fig2, axes2 = plt.subplots(1, show_count, figsize=(4 * show_count, 5))
        if show_count == 1:
            axes2 = [axes2]

        for j in range(show_count):
            subset_i, real_idx, conf = bhayanak_mistakes[j]
            img, true_label = test_dataset[real_idx]
            img_show = denormalize(img).permute(1, 2, 0).numpy()

            # Full probability breakdown
            model.eval()
            with torch.no_grad():
                output = model(img.unsqueeze(0).to(device))
                probs_single = torch.softmax(output, dim=1).cpu().numpy()[0]

            filepath = test_dataset.samples[real_idx][0]
            filename = os.path.basename(filepath)

            axes2[j].imshow(img_show)
            axes2[j].set_title(
                f"File: {filename}\n"
                f"Pred: {class_names[bhayanak_class_idx]} ({conf*100:.1f}%)\n"
                f"True: {class_names[adhbut_class_idx]}",
                fontsize=9, color='red'
            )
            axes2[j].axis('off')

            print(f"\nImage {j+1}: {filename}")
            for k in range(len(class_names)):
                marker = " <<<" if k == bhayanak_class_idx else ""
                print(f"  {class_names[k]}: {probs_single[k]*100:.1f}%{marker}")

        fig2.suptitle("Top 5 Adhbuta → Bhayanak Misclassifications", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(CONFIG["save_dir"], "adhbuta_as_bhayanak_top5.png"), dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print("\nNo Adhbuta images were misclassified as Bhayanak.")

all_wrong = []
for i in range(len(all_preds)):
    if all_preds[i] != adhbut_class_idx:
        all_wrong.append((i, adhbut_indices[i], all_preds[i], all_confs[i]))

all_wrong.sort(key=lambda x: -x[3])
show_all = min(5, len(all_wrong))

if show_all > 0:
    print(f"\n{'='*60}")
    print(f"TOP {show_all} OVERALL WORST ADHBUTA MISTAKES (ANY CLASS)")
    print(f"{'='*60}")

    fig3, axes3 = plt.subplots(1, show_all, figsize=(4 * show_all, 5))
    if show_all == 1:
        axes3 = [axes3]

    for j in range(show_all):
        subset_i, real_idx, pred_cls, conf = all_wrong[j]
        img, _ = test_dataset[real_idx]
        img_show = denormalize(img).permute(1, 2, 0).numpy()
        filename = os.path.basename(test_dataset.samples[real_idx][0])

        axes3[j].imshow(img_show)
        axes3[j].set_title(
            f"{filename}\n"
            f"Pred: {class_names[pred_cls]} ({conf*100:.1f}%)\n"
            f"True: {class_names[adhbut_class_idx]}",
            fontsize=9, color='red'
        )
        axes3[j].axis('off')

    fig3.suptitle("Top 5 Overall Worst Adhbuta Mistakes", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["save_dir"], "adhbuta_worst_mistakes.png"), dpi=150, bbox_inches='tight')
    plt.show()

print(f"\nAll plots saved to {CONFIG['save_dir']}")


Found Adhbuta class at index 0: adhbuta
Total Adhbuta test samples: 303

ADHBUTA TEST ACCURACY: 77.89% (236/303)

Misclassification Breakdown:
  Confused as Bhayanak: 20 times (6.6%)
  Confused as Bhibhatsa: 19 times (6.3%)
  Confused as Hasya: 9 times (3.0%)
  Confused as Karuna: 15 times (5.0%)
  Confused as Raudra: 3 times (1.0%)
  Confused as Veer: 1 times (0.3%)

Confidence Statistics:
  Correct predictions avg confidence: 83.5%
  Wrong predictions avg confidence:   27.0%

TOP 5 ADHBUTA IMAGES MISCLASSIFIED AS BHAYANAK

Image 1: test_0123.jpg
  adhbuta: 30.2%
  Bhayanak: 34.2% <<<
  Bhibhatsa: 4.3%
  Hasya: 9.3%
  Karuna: 4.0%
  Raudra: 5.8%
  Shanta: 5.8%
  Shringara: 4.8%
  Veer: 1.5%

Image 2: newiwe (2)_frame_0498.jpg
  adhbuta: 27.1%
  Bhayanak: 30.7% <<<
  Bhibhatsa: 4.4%
  Hasya: 9.4%
  Karuna: 8.6%
  Raudra: 2.1%
  Shanta: 3.1%
  Shringara: 11.6%
  Veer: 3.0%

Image 3: test_0145.jpg
  adhbuta: 15.3%
  Bhayanak: 30.1% <<<
  Bhibhatsa: 4.5%
  Hasya: 15.9%
  Karuna: 5.2%
  Ra

In [ ]:

!pip install -q grad-cam

import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2
from torchvision import transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Input and Output Folders
INPUT_FOLDER = "/content/drive/MyDrive/navarasa_results_new/image_tester"
OUTPUT_FOLDER = "/content/drive/MyDrive/navarasa_results_new/output"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
image_paths = [os.path.join(INPUT_FOLDER, f) for f in os.listdir(INPUT_FOLDER)
               if f.lower().endswith(valid_extensions)]

if len(image_paths) == 0:
    print(f" No images found in {INPUT_FOLDER}!")
else:
    print(f" Found {len(image_paths)} images. Starting automated analysis...")

def analyze_facial_rasa_with_cam(image_path, model, device, class_names):
    print(f"\nProcessing: {os.path.basename(image_path)}...")

    filename = os.path.basename(image_path).split('.')[0]
    SAVE_PATH = os.path.join(OUTPUT_FOLDER, f"{filename}_detailed_analysis.png")

    try:
        img = Image.open(image_path).convert('RGB')
    except Exception as e:
        print(f" Error loading {image_path}: {e}")
        return

    # Applying same validation transforms to them
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    img_tensor = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = F.softmax(outputs, dim=1)[0] * 100

    probs_np = probabilities.cpu().numpy()
    sorted_indices = np.argsort(probs_np)[::-1]

    top_class_idx = sorted_indices[0]
    top_class = class_names[top_class_idx]
    top_prob = probs_np[top_class_idx]

    # grad cam heat map
    target_layer = None
    for module in model.modules():
        if isinstance(module, torch.nn.Conv2d):
            target_layer = module

    if target_layer is None:
        print("Could not find a convolutional layer for Grad-CAM.")
        return

    cam = GradCAM(model=model, target_layers=[target_layer])
    targets = [ClassifierOutputTarget(top_class_idx)]
    grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0, :]

    rgb_img = np.array(img.resize((224, 224))) / 255.0
    cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    # Final fesult display
    fig = plt.figure(figsize=(16, 5))
    fig.patch.set_facecolor('#FFFFFF')

    ax1 = fig.add_subplot(1, 3, 1)
    ax1.imshow(img)
    ax1.axis('off')
    ax1.set_title("Original Image", fontsize=14, fontweight='bold', pad=15)

    ax2 = fig.add_subplot(1, 3, 2)
    ax2.imshow(cam_image)
    ax2.axis('off')
    ax2.set_title(f"Grad-CAM (Focus Area)", fontsize=14, fontweight='bold', color='#E84855', pad=15)
    ax2.text(0.5, -0.05, f"Predicted: {top_class.upper()} ({top_prob:.1f}%)", transform=ax2.transAxes, ha='center', fontsize=13, fontweight='bold', color='#E84855')

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.set_facecolor('#FFFFFF')

    display_names = [class_names[i].capitalize() for i in sorted_indices]
    display_probs = [probs_np[i] for i in sorted_indices]

    y_pos = np.arange(len(display_names))
    colors = ['#E84855' if i == 0 else '#2E86AB' for i in range(len(display_names))]

    bars = ax3.barh(y_pos, display_probs, color=colors, alpha=0.85, height=0.6)
    ax3.set_yticks(y_pos)
    ax3.set_yticklabels(display_names, fontsize=11)
    ax3.invert_yaxis()
    ax3.set_xlabel('Confidence (%)', fontsize=12)
    ax3.set_title('Detailed Confidence Meter', fontsize=14, fontweight='bold', pad=15)
    ax3.grid(axis='x', linestyle='--', alpha=0.5)

    for bar in bars:
        width = bar.get_width()
        ax3.text(width + 2, bar.get_y() + bar.get_height()/2.,
                f'{width:.1f}%', ha='left', va='center', fontsize=10)

    ax3.set_xlim(0, 110)
    plt.tight_layout()

    plt.savefig(SAVE_PATH, dpi=150, bbox_inches='tight', facecolor='#FFFFFF')
    print(f" Saved Analysis to: {SAVE_PATH}")

    plt.show()

try:
    classes = val_dataset.classes
except NameError:
    classes = ['adhbuta', 'bhayanak', 'bhibhatsa', 'hasya', 'karuna', 'raudra', 'shanta', 'shringara', 'veer']

for img_path in sorted(image_paths):
    analyze_facial_rasa_with_cam(img_path, model, device, classes)

 Found 7 images. Starting automated analysis...


NameError: name 'model' is not defined